# 8 — Idea Generation Agent (Parser → Generator Pipeline)

A two-agent creative pipeline: the first agent structures raw data into a usable format; the second generates novel ideas from that structure.

**What you'll learn**
- Custom state with nested Pydantic models in `TypedDict` fields
- Structuring the parser agent to return typed, validated data (not just strings)
- How the idea generator reads `parsed_content` from state without knowing how it was populated
- JSON extraction from LLM output — handling markdown fences and parse failures
- Building creative pipelines: separate the structure phase from the generation phase

**Companion examples:** 3-prospect-agent (same pipeline pattern, research domain), 13-structured-output (structured extraction focus)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Data models and state ───────────────────────────────────────────────────
# State holds nested Pydantic objects.
# The parser fills 'parsed_content'; the generator reads it to fill 'ideas'.
from typing import TypedDict

from pydantic import BaseModel, Field


class FreeTool(BaseModel):
    name: str
    category: str
    description: str


class ParsedContent(BaseModel):
    tools: list[FreeTool]
    categories: list[str]
    summary: str


class IdeaConcept(BaseModel):
    title: str
    description: str
    target_tools: list[str]
    value_proposition: str
    effort_level: str


class IdeaOutput(BaseModel):
    idea: IdeaConcept
    reasoning: str
    next_steps: list[str]


class AgentState(TypedDict):
    parsed_content: ParsedContent | None
    ideas: list[IdeaOutput] | None


print("State: [parser fills parsed_content] -> [generator fills ideas]")

In [ ]:
# ── 4. Parser node — structures raw data into typed form ──────────────────────
# The original script fetches a GitHub README over HTTP and parses markdown.
# Here we use inline data so the notebook is self-contained.
#
# Key insight: the parser's job is to STRUCTURE data, not generate ideas.
# This separation keeps each node focused and independently testable.
from langchain_openai import ChatOpenAI

_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

FREE_TOOLS = [
    {"name": "Google Colab",       "category": "Machine Learning",  "description": "Free cloud Jupyter notebooks with GPU access"},
    {"name": "Hugging Face",        "category": "Machine Learning",  "description": "Model hub with free inference API tier"},
    {"name": "GitHub Actions",      "category": "CI/CD",             "description": "Automated workflows up to 2000 min/month free"},
    {"name": "Vercel",              "category": "Hosting",           "description": "Free tier for frontend deployments with previews"},
    {"name": "Supabase",            "category": "Database",          "description": "Postgres + auth + storage with free tier"},
    {"name": "Cloudflare Workers",  "category": "Serverless",       "description": "100k free requests/day, runs at the edge"},
    {"name": "Upstash",             "category": "Database",          "description": "Serverless Redis, free tier with 10k req/day"},
    {"name": "Resend",              "category": "Email",             "description": "100 emails/day free, developer-friendly API"},
    {"name": "Clerk",               "category": "Authentication",    "description": "User auth with social login, free up to 5k MAU"},
    {"name": "Sentry",              "category": "Error Tracking",    "description": "Error monitoring, free 5k events/month"},
    {"name": "Posthog",             "category": "Analytics",         "description": "Product analytics, free up to 1M events/month"},
    {"name": "Trigger.dev",         "category": "Background Jobs",   "description": "Background job queue, free tier available"},
    {"name": "Render",              "category": "Hosting",           "description": "Free tier for web services (sleep on idle)"},
    {"name": "Cal.com",             "category": "Scheduling",        "description": "Open-source scheduling, free self-host or cloud"},
    {"name": "Liveblocks",          "category": "Collaboration",     "description": "Real-time collaboration infra, free 100 MAU"},
]


def parser_node(state: AgentState) -> dict:
    tools = [FreeTool(**t) for t in FREE_TOOLS]
    categories = sorted(set(t.category for t in tools))
    content = ParsedContent(
        tools=tools,
        categories=categories,
        summary=f"Found {len(tools)} free tools across {len(categories)} categories: {', '.join(categories)}",
    )
    print(f"  [parser] structured {len(tools)} tools in {len(categories)} categories")
    return {"parsed_content": content}


print("Parser node ready")

In [ ]:
# ── 5. Generator node — creates ideas from structured data ────────────────────
# The generator reads parsed_content from state.
# It doesn't know HOW the data was collected — just what's in it.
#
# JSON extraction pattern:
#   LLMs often wrap JSON in ```json fences — always strip them before json.loads().
#   Fall back to regex if the response has leading/trailing prose.
import json
import re

from langchain_core.messages import HumanMessage, SystemMessage


def generator_node(state: AgentState) -> dict:
    content = state["parsed_content"]
    tool_list = "\n".join(f"- {t.name} ({t.category}): {t.description}" for t in content.tools)

    print(f"  [generator] generating ideas from {len(content.tools)} tools...")

    resp = _llm.invoke([
        SystemMessage(
            "You are a product ideation expert. Given a list of free developer tools, generate 3-5 "
            "practical product ideas that combine 2-3 tools. Return a JSON array only, no prose. "
            "Each object must have: title, description, target_tools (array), value_proposition, "
            "effort_level (low/medium/high), reasoning, next_steps (array of 3 strings)."
        ),
        HumanMessage(f"Available free tools:\n{tool_list}\n\nGenerate 3-5 ideas:"),
    ])

    raw = resp.content.strip()

    # Strip ```json fences if present
    fence_match = re.search(r"```(?:json)?\s*(\[.*?\])\s*```", raw, re.DOTALL)
    if fence_match:
        raw = fence_match.group(1)
    else:
        arr_match = re.search(r"(?s)(\[.*\])", raw)
        if arr_match:
            raw = arr_match.group(1)

    ideas_data = json.loads(raw)
    ideas = [
        IdeaOutput(
            idea=IdeaConcept(
                title=d["title"],
                description=d["description"],
                target_tools=d["target_tools"],
                value_proposition=d["value_proposition"],
                effort_level=d["effort_level"],
            ),
            reasoning=d.get("reasoning", ""),
            next_steps=d.get("next_steps", []),
        )
        for d in ideas_data
        if isinstance(d, dict)
    ]
    print(f"  [generator] produced {len(ideas)} ideas")
    return {"ideas": ideas}


print("Generator node ready")

In [ ]:
# ── 6. Wire the graph and run ─────────────────────────────────────────────────
from langgraph.graph import END, START, StateGraph

graph = StateGraph(AgentState)
graph.add_node("parser", parser_node)
graph.add_node("generator", generator_node)
graph.add_edge(START, "parser")
graph.add_edge("parser", "generator")
graph.add_edge("generator", END)
app = graph.compile()

print("Running pipeline: parser -> generator\n")
result = app.invoke({"parsed_content": None, "ideas": None})

print(f"\n=== {len(result['ideas'])} IDEAS GENERATED ===")
for i, item in enumerate(result["ideas"], 1):
    idea = item.idea
    print(f"\n{i}. {idea.title} [{idea.effort_level} effort]")
    print(f"   {idea.description}")
    print(f"   Tools: {', '.join(idea.target_tools)}")
    print(f"   Value: {idea.value_proposition}")
    print(f"   Why: {item.reasoning[:120]}")

## Exercises

**Exercise 1 — Change the domain:** Replace `FREE_TOOLS` with tools from a domain you care about (marketing tools, design tools, data tools). Re-run. Do the generated ideas make sense for the new domain?

**Exercise 2 — Add a scoring node:** Add a third `scorer_node` that reads `state['ideas']` and ranks each idea 1-5 for feasibility. Wire it: `generator -> scorer -> END`.

**Exercise 3 — JSON extraction hardening:** Modify `generator_node` to return `{"ideas": []}` instead of raising an exception when `json.loads()` fails. Test by intentionally breaking the prompt.

**Exercise 4 — Fetch real data:** The original script fetches the full free-for-dev list from GitHub. Replace the inline `FREE_TOOLS` with a `requests.get()` to that URL and parse the markdown. Do the ideas improve with more variety?

## What's next

- **3-prospect-agent** — same two-agent pipeline applied to personalized outreach
- **13-structured-output** — deeper structured extraction with complex Pydantic schemas
- **22-plan-execute** — plan first, then execute each step with a separate agent call